## Pitch Visualization

In [14]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Datenset laden

In [15]:
data = pd.read_csv('../data/raw/DATASET CSV 11-02-2026.csv', sep=';')
data['br-pos'] = pd.to_numeric(data['br-pos'], errors='coerce')
data['annahme'] = pd.to_numeric(data['annahme'], errors='coerce')

# Überblick über die Daten schaffen

In [16]:
data.head()
data.shape
data.columns
data.info()
data.isna().sum()
unique_titel = data['titel_kurz_d'].unique()
len(unique_titel)
unique_titel[:20]

<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Columns: 874 entries, anr to nach_cockpit_e
dtypes: float64(482), int64(10), str(382)
memory usage: 4.7 MB


<StringArray>
[                     'Bundesverfassung der schweizerischen Eidgenossenschaft',
                                                            'Mass und Gewicht',
    'Gleichstellung der Juden und Naturalisierten mit Bezug auf Niederlassung',
                  'Stimmrecht der Niedergelassenen in Gemeindeangelegenheiten',
           'Besteuerung und zivilrechtliche Verhältnisse der Niedergelassenen',
               'Stimmrecht der Niedergelassenen in kantonalen Angelegenheiten',
                                                'Glaubens- und Kultusfreiheit',
                                         'Ausschliessung einzelner Strafarten',
                                              'Schutz des geistigen Eigentums',
                                        'Verbot der Lotterie und Hasardspiele',
                                            'Bundesverfassung (Totalrevision)',
 'Gesetz betreffend Feststellung und Beurkundung des Zivilstandes und die Ehe',
            'Gesetz über d

# Wrangling und Datacleaning

## Datatypes

In [25]:
data['datum'] = pd.to_datetime(data['datum'], format='%d.%m.%Y')
data['jahr'] = data['datum'].dt.year
data.replace([9999,999], np.nan, inplace=True)
data["d1e1"] = pd.to_numeric(data["d1e1"], errors='coerce')
data["d1e2"] = pd.to_numeric(data["d1e2"], errors='coerce')
data["d1e3"] = pd.to_numeric(data["d1e3"], errors='coerce')

## Mapping

### Thematische Gruppen der Abstimmungen
Fokus auf Einteilung durch Swissvote (in Abstimmung mit BFS), da es sich um eine inhaltliche Überarbeitung der ursprünglichen BFS Position geht. Deshalb werden d2 und d3 nicht berücksichtigt.

In [ ]:
data.head()

In [20]:
# Hauptgruppe (d1e1)
def assign_hauptgruppe(row):
    d1 = row['d1e1']
    mapping = {
        1: 'Staatsordnung',
        2: 'Aussenpolitik',
        3: 'Sicherheitspolitik',
        4: 'Wirtschaft',
        5: 'Landwirtschaft',
        6: 'Öffentliche Finanzen',
        7: 'Energie',
        8: 'Verkehr & Infrastruktur',
        9: 'Umwelt & Lebensraum',
        10: 'Sozial- und Gesellschaftspolitik',
        11: 'Bildung & Forschung',
        12: 'Kultur, Religion & Medien',
    }
    return mapping.get(d1, 'Andere')

def assign_untergruppen(row):
    d2 = row['d1e2']

    mapping = {
        1.1: 'Nationale Identität',
        1.2: 'Politisches System',
        1.3: 'Institutionen',
        1.4: 'Volksrechte',
        1.5: 'Föderalismus',
        1.6: 'Rechtsordnung',
        2.1: 'Aussenpolitische Grundhaltung',
        2.2: 'Europapolitik',
        2.3: 'Internationale Organisationen',
        2.4: 'Entwicklungszusammenarbeit',
        2.5: 'Staatsverträge',
        2.6: 'Aussenwirtschaftspolitik',
        2.7: 'Diplomatie',
        2.8: 'Auslandschweizer:innen',
        3.1: 'Öffentliche Sicherheit',
        3.2: 'Armee',
        3.3: 'Landesversorgung',
        4.1: 'Wirtschaftspolitik',
        4.2: 'Arbeit und Beschäftigung',
        4.3: 'Finanzwesen',
        4.4: 'Freizeit und Tourismus',
        5.1: 'Agrarpolitik',
        5.2: 'Tierische Produktion',
        5.3: 'Pflanzliche Produktion',
        5.4: 'Forstwirtschaft',
        5.5: 'Fischerei, Jagd, Haustiere',
        6.1: 'Steuerwesen',
        6.2: 'Finanzordnung',
        6.3: 'Öffentliche Ausgaben',
        6.4: 'Spar- und Sanierungsmassnahmen',
        7.1: 'Energiepolitik',
        7.2: 'Kernenergie',
        7.3: 'Wasserkraft',
        7.4: 'Alternativenergien',
        7.5: 'Erdöl, Gas',
        8.1: 'Verkehrspolitik',
        8.2: 'Strassenverkehr',
        8.3: 'Schienenverkehr',
        8.4: 'Luftverkehr',
        8.5: 'Schifffahrt',
        8.6: 'Post',
        8.7: 'Telekommunikation',
        9.1: 'Boden',
        9.2: 'Wohnen',
        9.3: 'Umwelt',
        10.1: 'Gesundheit',
        10.2: 'Sozialversicherungen',
        10.3: 'Gesellschaftsfragen',
        11.1: 'Bildungspolitik',
        11.2: 'Schulen',
        11.3: 'Hochschulen',
        11.4: 'Forschung',
        11.5: 'Berufsbildung',
        12.1: 'Kulturpolitik',
        12.2: 'Sprachpolitik',
        12.3: 'Religion, Kirchen',
        12.4: 'Sport',
        12.5: 'Medien und Kommunikation',
    }
    return mapping.get(d2, 'Andere')

# Kleinstgruppe (d1e3)
def assign_kleinstgruppe(row):
    d3 = row['d1e3']
    mapping = {
        1.21: 'Bundesverfassung',
        1.22: 'Verfassungsgebungsverfahren',
        1.23: 'Gesetzgebungsverfahren',
        1.24: 'Wahlsystem',
        1.31: 'Regierung, Verwaltung',
        1.32: 'Parlament',
        1.33: 'Gerichte',
        1.34: 'Nationalbank',
        1.41: 'Initiative',
        1.42: 'Referendum',
        1.43: 'Stimmrecht',
        1.51: 'Territorialfragen',
        1.52: 'Beziehungen Bund–Kantone',
        1.53: 'Aufgabenteilung',
        1.61: 'Internationales Recht',
        1.62: 'Grundrechte',
        1.63: 'Bürgerrecht',
        1.64: 'Privatrecht',
        1.65: 'Strafrecht',
        1.66: 'Datenschutz',
        2.11: 'Neutralität',
        2.12: 'Unabhängigkeit',
        2.13: 'Gute Dienste',
        2.21: 'EFTA',
        2.22: 'EU',
        2.23: 'EWR',
        2.24: 'Andere europäische Organisationen',
        2.31: 'UNO',
        2.32: 'Andere internationale Organisationen',
        2.61: 'Exportförderung',
        2.62: 'Zollwesen',
        3.11: 'Bevölkerungsschutz',
        3.12: 'Staatsschutz',
        3.13: 'Polizei',
        3.21: 'Armee (allgemein)',
        3.22: 'Militärorganisation',
        3.23: 'Rüstung',
        3.24: 'Militäranlagen',
        3.25: 'Dienstverweigerung, Zivildienst',
        3.26: 'Armeeabschaffung',
        3.27: 'Militärische Ausbildung',
        3.28: 'Internationale Einsätze',
        4.11: 'Konjunkturpolitik',
        4.12: 'Wettbewerbspolitik',
        4.13: 'Strukturpolitik',
        4.14: 'Preispolitik',
        4.15: 'Konsumentenschutz',
        4.16: 'Gesellschaftsrecht',
        4.21: 'Arbeitsbedingungen',
        4.22: 'Arbeitszeit',
        4.23: 'Sozialpartnerschaft',
        4.24: 'Beschäftigungspolitik',
        4.31: 'Geld- und Währungspolitik',
        4.32: 'Banken, Börsen, Versicherungen',
        4.41: 'Fremdenverkehr',
        4.42: 'Hotellerie und Gastgewerbe',
        4.43: 'Geldspiele',
        6.11: 'Steuerpolitik',
        6.12: 'Steuersystem',
        6.13: 'Direkte Steuern',
        6.14: 'Indirekte Steuern',
        8.11: 'Agglomerationsverkehr',
        8.12: 'Transitverkehr',
        8.21: 'Strassenbau',
        8.22: 'Schwerverkehr',
        8.31: 'Güterverkehr',
        8.32: 'Personenverkehr',
        9.11: 'Raumplanung',
        9.12: 'Bodenrecht',
        9.21: 'Mietwesen',
        9.22: 'Wohnungsbau, Wohneigentum',
        9.31: 'Umweltpolitik',
        9.32: 'Lärmschutz',
        9.33: 'Luftreinhaltung',
        9.34: 'Gewässerschutz',
        9.35: 'Bodenschutz',
        9.36: 'Abfälle',
        9.37: 'Natur- und Heimatschutz',
        9.38: 'Tierschutz',
        10.11: 'Gesundheitspolitik',
        10.12: 'Medizinforschung und -technik',
        10.13: 'Medikamente',
        10.14: 'Suchtmittel',
        10.15: 'Fortpflanzungsmedizin',
        10.21: 'AHV',
        10.22: 'Invalidenversicherung',
        10.23: 'Berufliche Vorsorge',
        10.24: 'Kranken- und Unfallversicherung',
        10.25: 'Mutterschaftsversicherung',
        10.26: 'Arbeitslosenversicherung',
        10.27: 'Erwerbsersatzordnung',
        10.28: 'Fürsorge',
        10.31: 'Migrations- und Integrationspolitik',
        10.32: 'Asylpolitik',
        10.33: 'Frauen und Gleichstellungspolitik',
        10.34: 'Familienpolitik',
        10.35: 'Kinder- und Jugendpolitik',
        10.36: 'Alterspolitik',
        10.37: 'Menschen mit Behinderungen',
        10.38: 'LGBTQIA+',
        11.41: 'Gentechnologie',
        11.42: 'Tierversuche',
        12.51: 'Medienpolitik',
        12.52: 'Presse',
        12.53: 'Radio, Fernsehen, Elektronische Medien',
        12.54: 'Medienfreiheit',
    }
    return mapping.get(d3, None)


# Anwendung
data['hauptgruppe'] = data.apply(assign_hauptgruppe, axis=1)
data['untergruppe'] = data.apply(assign_untergruppen, axis=1)
data['kleinstgruppe'] = data.apply(assign_kleinstgruppe, axis=1)


print("N hauptgruppe", data['hauptgruppe'].value_counts(dropna=False))
print("N untergruppe", data['untergruppe'].value_counts(dropna=False))
print("N kleinstgruppe", data['kleinstgruppe'].value_counts(dropna=False))

N hauptgruppe hauptgruppe
Sozial- und Gesellschaftspolitik    142
Staatsordnung                       107
Wirtschaft                           86
Öffentliche Finanzen                 71
Verkehr & Infrastruktur              58
Sicherheitspolitik                   53
Umwelt & Lebensraum                  49
Landwirtschaft                       38
Aussenpolitik                        34
Bildung & Forschung                  25
Energie                              24
Kultur, Religion & Medien            21
Name: count, dtype: int64
N untergruppe untergruppe
Sozialversicherungen              61
Wirtschaftspolitik                50
Steuerwesen                       42
Gesellschaftsfragen               42
Armee                             40
Gesundheit                        39
Rechtsordnung                     35
Institutionen                     32
Strassenverkehr                   29
Volksrechte                       21
Arbeit und Beschäftigung          19
Umwelt                            1

In [24]:
data[['titel_kurz_d', 'd1e1', 'd1e2', 'd1e3', 'hauptgruppe', 'untergruppe', 'kleinstgruppe']].sample(20, random_state=42)


,titel_kurz_d,d1e1,d1e2,d1e3,hauptgruppe,untergruppe,kleinstgruppe
120,Neuordnung der militärischen Ausbildung,3,3.2,3.27,Sicherheitspolitik,Armee,Militärische Ausbildung
247,Tierschutzartikel,9,9.3,9.38,Umwelt & Lebensraum,Umwelt,Tierschutz
324,Energieartikel,7,7.1,NaN,Energie,Energiepolitik,NaN
204,Natur- und Heimatschutz-Artikel,9,9.3,9.37,Umwelt & Lebensraum,Umwelt,Natur- und Heimatschutz
603,Initiative «Schluss mit der MwSt-Diskriminieru...,6,6.1,6.14,Öffentliche Finanzen,Steuerwesen,Indirekte Steuern
356,Asylgesetz,10,10.3,10.32,Sozial- und Gesellschaftspolitik,Gesellschaftsfragen,Asylpolitik
81,Initiative für ein Verbot der Spielbanken,4,4.1,4.13,Wirtschaft,Wirtschaftspolitik,Strukturpolitik
54,Vereinheitlichung des Strafrechts,1,1.6,1.65,Staatsordnung,Rechtsordnung,Strafrecht
512,Gesetz über den Zivilschutz,3,3.1,3.11,Sicherheitspolitik,Öffentliche Sicherheit,Bevölkerungsschutz
572,Initiative «Für den Schutz vor Waffengewalt»,1,1.6,NaN,Staatsordnung,Rechtsordnung,NaN


### Rechtsform

In [26]:
rechtsform_map = {
    1: 'Obligatorisches Referendum',
    2: 'Fakultatives Referendum',
    3: 'Volksinitiative',
    4: 'Direkter Gegenentwurf',
    5: 'Stichfrage'
}

data['rechtsform_name'] = data['rechtsform'].map(rechtsform_map)
print(data['rechtsform_name'])

0      Obligatorisches Referendum
1      Obligatorisches Referendum
2      Obligatorisches Referendum
3      Obligatorisches Referendum
4      Obligatorisches Referendum
                  ...            
703               Volksinitiative
704               Volksinitiative
705       Fakultatives Referendum
706               Volksinitiative
707       Fakultatives Referendum
Name: rechtsform_name, Length: 708, dtype: str


In [ ]:
### Katnonsangaben
In alten

#### Sprachregion

In [ ]:
# Sprachregionen zuordnen
sprachregion_map = {
    'ZH': 'Deutschschweiz', 'BE': 'Deutschschweiz', 'LU': 'Deutschschweiz',
    'UR': 'Deutschschweiz', 'SZ': 'Deutschschweiz', 'OW': 'Deutschschweiz',
    'NW': 'Deutschschweiz', 'GL': 'Deutschschweiz', 'ZG': 'Deutschschweiz',
    'FR': 'Romandie', 'SO': 'Deutschschweiz', 'BS': 'Deutschschweiz',
    'BL': 'Deutschschweiz', 'SH': 'Deutschschweiz', 'AR': 'Deutschschweiz',
    'AI': 'Deutschschweiz', 'SG': 'Deutschschweiz', 'GR': 'Deutschschweiz',
    'AG': 'Deutschschweiz', 'TG': 'Deutschschweiz', 'TI': 'Tessin',
    'VD': 'Romandie', 'VS': 'Romandie', 'NE': 'Romandie',
    'GE': 'Romandie', 'JU': 'Romandie'
}

# Visualisierungen

### Regierungstreue des Volks im Vergleich zum Bundesrat (pro 10 Jahres Rhytmus) Figure.1

In [ ]:
plot_data = plot_data[plot_data['rechtsform_group'] != 'Direkter Gegenentwurf']

counts = plot_data.groupby('dekade').size()
valid_dekaden = counts[counts >= 5].index

plot_data = plot_data[plot_data['dekade'].isin(valid_dekaden)]

plt.figure(figsize=(12,6))

sns.lineplot(
    data=plot_data,
    x='dekade',
    y='regierungstreue',
    hue='rechtsform_group',
    estimator='mean',
    errorbar=None,
    marker='o',
    linewidth=2.5
)

plt.title('Folgt das Volk dem Bundesrat? Regierungskonformität seit 1870')
plt.legend(title='Abstimmungstyp', loc='upper left')
plt.xlabel('Dekade')
plt.ylabel('Anteil regierungskonformer Abstimmungen')
plt.ylim(0,1)

plt.grid(alpha=0.3)

plt.show()

### Wo vertraut das Volk dem Bundesrat am meisten Figure.2

In [ ]:
plt.figure(figsize=(10,7))

topic_order = (
    plot_data.groupby('Hauptgruppe')['regierungstreue']
    .mean()
    .sort_values()
    .index
)

sns.barplot(
    data=plot_data,
    x='regierungstreue',
    y='Hauptgruppe',
    order=topic_order,
    estimator='mean',
    errorbar=None
)

plt.title('Wo folgt das Volk dem Bundesrat am häufigsten?')
plt.xlabel('Anteil regierungskonformer Abstimmungen')
plt.ylabel('Politikbereich')

plt.xlim(0,1)
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()



### Wo vertraut das Volk dem Bundesrat am meisten Figure.3



In [ ]:
plot_data['abweichung'] = 1 - plot_data['regierungstreue']
plt.figure(figsize=(10,7))

topic_order = (
    plot_data.groupby('Hauptgruppe')['abweichung']
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.barplot(
    data=plot_data,
    x='abweichung',
    y='Hauptgruppe',
    order=topic_order,
    estimator='mean',
    errorbar=None
)

plt.title('Wo widerspricht das Volk dem Bundesrat am häufigsten?')
plt.xlabel('Anteil nicht regierungskonformer Abstimmungen')
plt.ylabel('Politikbereich')
plt.xlim(0,1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()



### Wo vertraut das Volk dem Bundesrat am meisten Figure.4

In [ ]:
topic_summary = (
    plot_data.groupby('Hauptgruppe')['regierungstreue']
    .mean()
    .reset_index()
)

topic_summary['abweichung'] = 1 - topic_summary['regierungstreue']
topic_summary = topic_summary.sort_values('regierungstreue')

topic_long = topic_summary.melt(
    id_vars='Hauptgruppe',
    value_vars=['regierungstreue', 'abweichung'],
    var_name='Typ',
    value_name='Anteil'
)

plt.figure(figsize=(11,7))

sns.barplot(
    data=topic_long,
    x='Anteil',
    y='Hauptgruppe',
    hue='Typ'
)

plt.title('Folgt das Volk dem Bundesrat oder widerspricht es ihm?')
plt.xlabel('Anteil der Abstimmungen')
plt.ylabel('Politikbereich')
plt.xlim(0,1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()




### Gibt es regionale Unterschiede darin, wie stark Kantone der Empfehlung des Bundesrats folgen? (Boxplot) 
ungeeignet habe ihn dennoch mal drin gelassen

In [ ]:
plt.figure(figsize=(14, 7))

sns.boxplot(
    data=kanton_df,
    x='Kanton',
    y='Ja_Anteil'
)

plt.title('Verteilung der Ja-Anteile nach Kanton')
plt.xlabel('Kanton')
plt.ylabel('Ja-Stimmen-Anteil')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()



### Wie häufig folgen Kantone dem Bundesrat?


In [ ]:
# FacetGrid nach Sprachregion
g = sns.FacetGrid(
    kanton_summary,
    col="Sprachregion",
    col_wrap=3,
    height=5,
    sharex=True,
    sharey=False
)

g.map_dataframe(
    sns.barplot,
    x="kanton_regierungstreue",
    y="Kanton",
    order=kanton_summary.sort_values('kanton_regierungstreue')['Kanton']
)

g.set_axis_labels("Anteil regierungskonformer Abstimmungen", "Kanton")
g.set_titles("{col_name}")

g.fig.suptitle(
    "Kantonale Regierungstreue nach Sprachregion",
    y=1.03,
    fontsize=14
)

plt.xlim(0,1)
plt.tight_layout()
plt.show()


# Plot
plt.figure(figsize=(14, 8))

sns.barplot(
    data=kanton_summary,
    x='kanton_regierungstreue',
    y='Kanton',
    hue='Sprachregion'
)

plt.title('Kantonale Regierungstreue: Wie häufig folgen Kantone dem Bundesrat?')
plt.xlabel('Anteil kantonal regierungskonformer Abstimmungen')
plt.ylabel('Kanton')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Visualisierung Nr. 3 – Der Röstigraben des Vertrauens

In [ ]:
# FacetGrid nach Sprachregion
g = sns.FacetGrid(
    kanton_summary,
    col="Sprachregion",
    col_wrap=3,
    height=5,
    sharex=True,
    sharey=False
)

g.map_dataframe(
    sns.barplot,
    x="kanton_regierungstreue",
    y="Kanton",
    order=kanton_summary.sort_values('kanton_regierungstreue')['Kanton']
)

g.set_axis_labels("Anteil regierungskonformer Abstimmungen", "Kanton")
g.set_titles("{col_name}")

g.fig.suptitle(
    "Der Röstigraben des Vertrauens: Kantonale Regierungstreue nach Sprachregion",
    y=1.03,
    fontsize=14
)

for ax in g.axes.flat:
    ax.set_xlim(0, 1)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()




### Gibt es Regionale Unterschiede darin,wie stark die Kantone der Empfehlung des Bundesrates folgen?? Figure.5

In [ ]:
plt.figure(figsize=(16, 8))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    center=50
)

plt.title("Durchschnittlicher Ja-Stimmen-Anteil nach Hauptgruppe und Kanton")
plt.xlabel("Kanton")
plt.ylabel("Hauptgruppe")
plt.tight_layout()
plt.show()